In [10]:
import numpy as np
from scipy.linalg import orth
from scipy.sparse.linalg import LinearOperator
# 生成稀疏矩阵 A，基于给定的特征值和特征向量


def generate_sparse_matrix(eigenvalues, eigenvectors):
    """
    根据给定的特征值和特征向量生成稀疏矩阵 A。
    特征值 λ 和特征向量 v 满足 A * v = λ * v。
    :param eigenvalues: 特征值数组 (一维)
    :param eigenvectors: 特征向量矩阵（每列为一个特征向量，二维数组）
    :return: 返回生成的矩阵 A
    """
    # 构造特征值的对角矩阵 D
    D = np.diag(eigenvalues)  # D 是一个对角矩阵，对角线元素为特征值
    # 特征向量矩阵 P 为每一列为一个特征向量
    P = eigenvectors  # 假设特征向量矩阵 P 是正交的
    # 通过 P * D * P^T 生成原始矩阵 A
    A = P @ D @ P.T  # A = P * D * P^T 是一种常见的矩阵分解方式
    return A
# Arnoldi 算法（Krylov 子空间生成器）


def arnoldi_iteration(A, k):
    """
    使用 Arnoldi 算法进行 Krylov 子空间迭代，生成矩阵的近似特征向量。
    通过 Arnoldi 算法，逐步生成一个正交的基向量组 {q1, q2, ..., qk}，
    用来近似矩阵 A 的特征值和特征向量。
    :param A: 输入矩阵（稀疏矩阵）
    :param k: Krylov 子空间的维度（即生成的基向量数量）
    :return: Arnoldi 基向量矩阵 Q 和上 Hessenberg 矩阵 H
    """
    n = A.shape[0]  # 获取矩阵 A 的大小
    # 初始化基向量矩阵 Q 和 Hessenberg 矩阵 H
    Q = np.zeros((n, k + 1))  # 用于存储基向量的矩阵
    H = np.zeros((k + 1, k))  # 用于存储上 Hessenberg 矩阵 H
    # 随机生成初始向量 q
    q = np.random.rand(n)  # 随机生成一个向量 q
    q = q / np.linalg.norm(q)  # 将向量 q 归一化，使其模长为 1
    Q[:, 0] = q  # 将归一化后的 q 作为第一个基向量
    # Krylov 子空间的生成
    for j in range(k):
        # 计算 A 作用于基向量 Q[:, j]，得到向量 v
        v = A @ Q[:, j]  # v = A * qj
        for i in range(j + 1):
            # 计算 v 与 Q 的内积，得到 H 矩阵的元素 H[i, j]
            H[i, j] = np.dot(Q[:, i], v)  # H[i, j] = q_i^T * v
            # 通过 Gram-Schmidt 正交化过程更新 v，减去与 Q 各列的投影
            v -= H[i, j] * Q[:, i]  # v = v - H[i, j] * q_i
        # 计算 v 的范数，并更新 H 的下三角元素
        H[j + 1, j] = np.linalg.norm(v)  # H[j + 1, j] = ||v||
        if H[j + 1, j] != 0 and j + 1 < n:
            # 归一化 v 得到新的基向量 Q[:, j + 1]
            Q[:, j + 1] = v / H[j + 1, j]  # q_(j+1) = v / ||v||
    return Q[:, :-1], H[:-1, :]  # 返回正交基向量矩阵 Q 和上 Hessenberg 矩阵 H
# QR 算法求解特征值和特征向量


def qr_algorithm(H, max_iter=1000, tol=1e-6):
    """
    使用 QR 算法计算上 Hessenberg 矩阵 H 的特征值和特征向量。
    手动实现 QR 分解（不使用任何库函数），并通过迭代得到特征值和特征向量。
    QR 算法的基本步骤：
    1. 进行 QR 分解：将矩阵 H 分解为 Q 和 R，其中 Q 为正交矩阵，R 为上三角矩阵。
       H = Q * R
    2. 更新 H 为 H = R * Q
    3. 重复进行上述步骤，直到 H 收敛为对角矩阵。
    :param H: 输入的上 Hessenberg 矩阵
    :param max_iter: 最大迭代次数
    :param tol: 收敛阈值，判断迭代是否达到精度
    :return: 计算得到的特征值和特征向量
    """
    n = H.shape[0]  # 获取矩阵 H 的维度（假设为 n x n）
    # 迭代过程，进行 QR 分解并更新 H
    for i in range(max_iter):
        # Step 1: 使用 Gram-Schmidt 正交化过程进行 QR 分解
        Q = np.zeros((n, n))  # 正交矩阵 Q
        R = np.zeros((n, n))  # 上三角矩阵 R
        # Gram-Schmidt 正交化过程
        for j in range(n):
            # 第 j 列作为当前向量 v
            v = H[:, j]
            # 对 v 进行正交化，使其与之前的基向量正交
            for k in range(j):
                # 计算 Q[:, k] 和 v 的点积，得到 R[k, j]
                R[k, j] = np.dot(Q[:, k], v)  # R[k, j] = q_k^T * v
                # 从 v 中减去与 Q[:, k] 的投影
                v -= R[k, j] * Q[:, k]  # v = v - R[k, j] * q_k
            # 归一化 v，得到新的基向量 Q[:, j]
            norm_v = np.linalg.norm(v)  # 计算向量 v 的范数
            if norm_v > tol:  # 检查范数是否大于小阈值
                R[j, j] = norm_v  # 计算 R[j, j] = ||v||
                Q[:, j] = v / R[j, j]  # q_j = v / ||v||，归一化向量 v
            else:
                # 如果范数小于阈值，则跳过该列的处理，R[j, j] 设置为一个小的数值
                R[j, j] = 1e-10  # 或者选择一个适当的小数值
                Q[:, j] = v  # 直接将 v 作为 Q[:, j]
        # Step 2: 计算新的 H = R * Q
        H_new = np.dot(R, Q)  # H_new = R * Q
        # Step 3: 检查收敛性
        # 如果 H 的非对角部分的元素都非常小，则认为已经收敛，停止迭代
        if np.all(np.abs(H_new - np.diag(np.diag(H_new))) < tol):
            break
        # 更新 H 为新的 H
        H = H_new
    # 返回 H 的对角线元素作为特征值，Q 为特征向量
    return np.diag(H), Q  # 特征值为对角线元素，特征向量为 Q 矩阵
# 计算两个向量的余弦相似度


def cosine_similarity(v1, v2):
    """
    计算两个向量的余弦相似度。
    余弦相似度公式为：
    cos(θ) = (v1 • v2) / (||v1|| * ||v2||)
    其中，• 表示向量点积，||v|| 表示向量的范数（模长）。
    :param v1: 第一个向量
    :param v2: 第二个向量
    :return: 返回余弦相似度
    """
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
# 验证特征向量与矩阵的关系 A * v = lambda * v


def validate_eigenvector(A, eigenvalue, eigenvector):
    """
    验证特征向量是否满足矩阵与特征值的关系 A * v = lambda * v。
    通过计算 A * v 和 λ * v 的差异来检查特征向量的正确性。
    1. 计算 A * v
    2. 计算 λ * v
    3. 计算误差 || A * v - λ * v ||，理论上误差应该很小。
    :param A: 矩阵
    :param eigenvalue: 特征值
    :param eigenvector: 特征向量
    :return: 返回误差（A * v - lambda * v 的范数）
    """
    # 计算 A 作用于特征向量 v
    Av = A @ eigenvector  # A * v
    # 计算 λ 作用于特征向量 v
    lambda_v = eigenvalue * eigenvector  # lambda * v
    # 计算两者之间的误差
    error = np.linalg.norm(Av - lambda_v)  # 误差 || A * v - lambda * v ||
    return error


# 主程序
if __name__ == "__main__":
    # 已知特征值和特征向量
    eigenvalues = np.array([])
    N = 10000
    n = 10
    for i in range(0, n):
        eigenvalues = np.append(eigenvalues, i*0.52)
    for i in range(n, N):
        eigenvalues = np.append(eigenvalues, 0)
    eigenvectors = orth(np.random.rand(N, N))  # 随机生成一个正交的特征向量矩阵
    A = generate_sparse_matrix(eigenvalues, eigenvectors)  # 根据特征值和特征向量生成稀疏矩阵 A
    # 使用 Arnoldi 算法提取 Krylov 子空间
    k = n   # 设置 Krylov 子空间的维度
    Q, H = arnoldi_iteration(A, k)  # 获取基向量矩阵 Q 和上 Hessenberg 矩阵 H
    # 使用 QR 算法计算 H 的特征值和特征向量
    estimated_eigenvalues, estimated_eigenvectors = qr_algorithm(
        H)  # 估计的特征值和特征向量
    # 通过 Arnoldi 基向量获取特征向量
    estimated_eigenvectors = Q @ estimated_eigenvectors  # 通过 Q 获取估计的特征向量
    # 输出计算结果
    print("原始特征值：", sorted(eigenvalues, reverse=True))  # 输出原始特征值
    print("估计特征值：", sorted(estimated_eigenvalues, reverse=True))  # 输出估计特征值
    print("估计特征向量（第一个特征向量）：", estimated_eigenvectors[:, 0])  # 输出第一个特征向量
    # 计算估计特征向量与原始特征向量的余弦相似度
    similarity = cosine_similarity(
        eigenvectors[:, 0], estimated_eigenvectors[:, 0])  # 计算余弦相似度
    print(f"原始特征向量与估计特征向量的余弦相似度：{similarity:.4f}")
    # 验证特征向量与矩阵的关系（A * v = lambda * v）
    eigenvalue = eigenvalues[0]  # 假设第一个特征值对应第一个特征向量
    eigenvector = eigenvectors[:, 0]  # 获取第一个特征向量
    error = validate_eigenvector(A, eigenvalue, eigenvector)  # 验证误差
    print(f"特征向量验证误差（A * v - lambda * v）：{error:.4e}")

原始特征值： [np.float64(4.68), np.float64(4.16), np.float64(3.64), np.float64(3.12), np.float64(2.6), np.float64(2.08), np.float64(1.56), np.float64(1.04), np.float64(0.52), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)